# SQL Analysis

This notebook demonstrates SQL-based analysis of 4,511 English-language Steam reviews for *Don't Starve Together*.

The processed review data was loaded into a local SQLite database with three normalized tables:

- `reviews` stores review-level information
- `themes` stores the theme lookup table
- `review_themes` represents the many-to-many relationship between reviews and themes

The analysis focuses on data quality, monthly review trends, playtime segmentation, and review themes.

### SQL techniques demonstrated

- `GROUP BY`
- `CASE WHEN`
- Conditional aggregation
- Common Table Expressions (CTEs)
- `JOIN` and `LEFT JOIN`
- `LAG`
- `RANK` and `DENSE_RANK`
- Running totals
- Rolling averages
- Many-to-many relationships

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

database_path = (
    project_root
    / "data"
    / "database"
    / "dst_reviews.db"
)

tables_dir = project_root / "outputs" / "tables"
figures_dir = project_root / "outputs" / "figures"

tables_dir.mkdir(
    parents=True,
    exist_ok=True,
)

figures_dir.mkdir(
    parents=True,
    exist_ok=True,
)

connection = sqlite3.connect(database_path)

print("Database:", database_path)

Database: /Users/zhouzhou/Desktop/dst-review-analysis/data/database/dst_reviews.db


## 1. Database Overview

The SQLite database uses a normalized structure rather than storing all information in a single wide table.

In [2]:
schema_query = """
SELECT
    name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

pd.read_sql_query(
    schema_query,
    connection,
)

,table_name
0,review_themes
1,reviews
2,themes


In [3]:
row_count_query = """
SELECT
    'reviews' AS table_name,
    COUNT(*) AS row_count
FROM reviews

UNION ALL

SELECT
    'themes',
    COUNT(*)
FROM themes

UNION ALL

SELECT
    'review_themes',
    COUNT(*)
FROM review_themes;
"""

pd.read_sql_query(
    row_count_query,
    connection,
)

,table_name,row_count
0,reviews,4511
1,themes,9
2,review_themes,3062


## 2. SQL Data Quality Checks

Before performing the analysis, SQL checks were used to verify primary-key uniqueness, missing values, invalid values, and referential integrity.

In [4]:
quality_query = """
WITH quality_checks AS (

    SELECT
        'Duplicate recommendation IDs' AS check_name,
        COUNT(*) - COUNT(DISTINCT recommendation_id)
            AS issue_count
    FROM reviews

    UNION ALL

    SELECT
        'Missing or empty review text',
        COUNT(*)
    FROM reviews
    WHERE review_text IS NULL
       OR TRIM(review_text) = ''

    UNION ALL

    SELECT
        'Invalid recommendation values',
        COUNT(*)
    FROM reviews
    WHERE recommended NOT IN (0, 1)
       OR recommended IS NULL

    UNION ALL

    SELECT
        'Negative playtime values',
        COUNT(*)
    FROM reviews
    WHERE playtime_at_review_hours < 0
       OR playtime_forever_hours < 0

    UNION ALL

    SELECT
        'Orphaned review-theme relationships',
        COUNT(*)
    FROM review_themes AS rt
    LEFT JOIN reviews AS r
        ON rt.recommendation_id
           = r.recommendation_id
    WHERE r.recommendation_id IS NULL
)

SELECT
    check_name,
    issue_count,

    CASE
        WHEN issue_count = 0 THEN 'PASS'
        ELSE 'REVIEW'
    END AS status

FROM quality_checks;
"""

quality_results = pd.read_sql_query(
    quality_query,
    connection,
)

quality_results

,check_name,issue_count,status
0,Duplicate recommendation IDs,0,PASS
1,Missing or empty review text,0,PASS
2,Invalid recommendation values,0,PASS
3,Negative playtime values,0,PASS
4,Orphaned review-theme relationships,0,PASS


All major SQL data-quality checks passed. The 4,511 review IDs are unique, no invalid recommendation values were found, playtime values are non-negative, and the review-theme relationship table contains no orphaned review references.

## 3. Monthly Review Metrics

Monthly review metrics were calculated using aggregation and SQL window functions. `LAG` was used to compare each month with the previous month, while `RANK` was used to rank months by review volume.

In [5]:
monthly_query = """
WITH monthly_base AS (
    SELECT
        review_month,

        COUNT(*) AS total_reviews,

        SUM(
            CASE
                WHEN recommended = 1 THEN 1
                ELSE 0
            END
        ) AS positive_reviews,

        SUM(
            CASE
                WHEN recommended = 0 THEN 1
                ELSE 0
            END
        ) AS negative_reviews,

        ROUND(
            100.0 * SUM(recommended) / COUNT(*),
            2
        ) AS positive_rate

    FROM reviews

    GROUP BY review_month
),

monthly_metrics AS (
    SELECT
        *,

        LAG(positive_rate) OVER (
            ORDER BY review_month
        ) AS previous_month_positive_rate,

        RANK() OVER (
            ORDER BY total_reviews DESC
        ) AS review_volume_rank

    FROM monthly_base
)

SELECT
    review_month,
    total_reviews,
    positive_reviews,
    negative_reviews,
    positive_rate,
    previous_month_positive_rate,

    ROUND(
        positive_rate
        - previous_month_positive_rate,
        2
    ) AS positive_rate_change_pp,

    review_volume_rank

FROM monthly_metrics

ORDER BY review_month;
"""

monthly_metrics = pd.read_sql_query(
    monthly_query,
    connection,
)

monthly_metrics

,review_month,total_reviews,positive_reviews,negative_reviews,positive_rate,previous_month_positive_rate,positive_rate_change_pp,review_volume_rank
0,2025-11,372,366,6,98.39,NaN,NaN,8
1,2025-12,484,437,47,90.29,98.39,-8.10,4
2,2026-01,421,381,40,90.50,90.29,0.21,5
3,2026-02,402,359,43,89.30,90.50,-1.20,7
4,2026-03,365,330,35,90.41,89.30,1.11,9
5,2026-04,498,453,45,90.96,90.41,0.55,3
6,2026-05,418,377,41,90.19,90.96,-0.77,6
7,2026-06,799,722,77,90.36,90.19,0.17,1
8,2026-07,699,623,76,89.13,90.36,-1.23,2
9,2026-08,53,45,8,84.91,89.13,-4.22,10


In [6]:
monthly_metrics.to_csv(
    tables_dir / "sql_monthly_metrics.csv",
    index=False,
)

### Findings

The positive rate remained close to 90% across most complete months.

June 2026 had the highest review volume with 799 English-language reviews. July remained high with 699 reviews.

The positive rate decreased from 90.36% in June to 89.13% in July, a decline of 1.23 percentage points.

November 2025 and August 2026 are partial months and should not be directly compared with complete months.

## 4. Review Segmentation by Playtime

`CASE WHEN` was used to divide reviews into five groups based on the reviewer's playtime when the review was submitted.

In [7]:
segment_query = """
WITH segmented_reviews AS (
    SELECT
        recommendation_id,
        recommended,
        review_word_count,
        votes_up,

        CASE
            WHEN playtime_at_review_hours < 2
                THEN 'Under 2 hours'

            WHEN playtime_at_review_hours < 10
                THEN '2 to under 10 hours'

            WHEN playtime_at_review_hours < 50
                THEN '10 to under 50 hours'

            WHEN playtime_at_review_hours < 200
                THEN '50 to under 200 hours'

            ELSE '200 hours or more'
        END AS playtime_segment,

        CASE
            WHEN playtime_at_review_hours < 2 THEN 1
            WHEN playtime_at_review_hours < 10 THEN 2
            WHEN playtime_at_review_hours < 50 THEN 3
            WHEN playtime_at_review_hours < 200 THEN 4
            ELSE 5
        END AS segment_order

    FROM reviews
)

SELECT
    playtime_segment,

    COUNT(*) AS total_reviews,

    ROUND(
        100.0 * SUM(recommended) / COUNT(*),
        2
    ) AS positive_rate,

    ROUND(
        AVG(review_word_count),
        2
    ) AS average_review_words,

    ROUND(
        100.0
        * AVG(
            CASE
                WHEN votes_up > 0 THEN 1.0
                ELSE 0.0
            END
        ),
        2
    ) AS helpful_vote_rate

FROM segmented_reviews

GROUP BY
    segment_order,
    playtime_segment

ORDER BY segment_order;
"""

player_segments = pd.read_sql_query(
    segment_query,
    connection,
)

player_segments

,playtime_segment,total_reviews,positive_rate,average_review_words,helpful_vote_rate
0,Under 2 hours,262,58.78,19.72,29.39
1,2 to under 10 hours,1062,88.32,19.93,16.29
2,10 to under 50 hours,1393,92.03,19.66,15.87
3,50 to under 200 hours,1033,95.55,23.67,14.23
4,200 hours or more,761,96.19,29.65,15.90


In [8]:
player_segments.to_csv(
    tables_dir / "sql_player_segments.csv",
    index=False,
)

### Findings

Recommendation rate increased strongly across playtime groups.

Reviews submitted before two hours of playtime had a positive rate of only 58.78%, compared with 96.19% among reviews submitted after 200 hours.

This association suggests that negative feedback is concentrated in the early player experience. However, the result does not establish that additional playtime causes higher satisfaction. Players who enjoy the game are also more likely to continue playing.

## 5. Theme Analysis with SQL JOINs

The theme analysis uses a normalized many-to-many relationship.

`reviews` is joined to `review_themes`, which is then joined to the `themes` lookup table.

In [9]:
theme_query = """
WITH totals AS (
    SELECT
        COUNT(*) AS total_reviews,

        SUM(
            CASE
                WHEN recommended = 0 THEN 1
                ELSE 0
            END
        ) AS total_negative_reviews

    FROM reviews
),

theme_metrics AS (
    SELECT
        t.theme_name,

        COUNT(
            DISTINCT r.recommendation_id
        ) AS theme_review_count,

        SUM(
            CASE
                WHEN r.recommended = 0 THEN 1
                ELSE 0
            END
        ) AS negative_theme_reviews

    FROM reviews AS r

    INNER JOIN review_themes AS rt
        ON r.recommendation_id
           = rt.recommendation_id

    INNER JOIN themes AS t
        ON rt.theme_id = t.theme_id

    GROUP BY t.theme_name
)

SELECT
    tm.theme_name,
    tm.theme_review_count,
    tm.negative_theme_reviews,

    ROUND(
        100.0
        * tm.negative_theme_reviews
        / tm.theme_review_count,
        2
    ) AS negative_rate_within_theme,

    ROUND(
        (
            1.0
            * tm.negative_theme_reviews
            / tm.theme_review_count
        )
        /
        (
            1.0
            * totals.total_negative_reviews
            / totals.total_reviews
        ),
        2
    ) AS negative_rate_lift

FROM theme_metrics AS tm

CROSS JOIN totals

ORDER BY negative_rate_lift DESC;
"""

theme_metrics = pd.read_sql_query(
    theme_query,
    connection,
)

theme_metrics

,theme_name,theme_review_count,negative_theme_reviews,negative_rate_within_theme,negative_rate_lift
0,Time and value,18,12,66.67,7.19
1,Repetition and boredom,100,41,41.00,4.42
2,Progression and clarity,116,38,32.76,3.54
3,Onboarding and guidance,133,39,29.32,3.16
4,Difficulty and punishment,160,27,16.88,1.82
5,Learning and challenge,166,13,7.83,0.85
6,Art and visual style,130,10,7.69,0.83
7,Social and multiplayer,765,51,6.67,0.72
8,Fun and enjoyment,1474,68,4.61,0.50


In [10]:
theme_metrics.to_csv(
    tables_dir / "sql_theme_metrics.csv",
    index=False,
)

### Findings

Several themes were disproportionately associated with negative reviews.

- **Time and value** had the highest negative-rate lift at 7.19, although it appeared in only 18 reviews, so the result should be interpreted cautiously.
- **Repetition and boredom** had a negative-rate lift of 4.42.
- **Progression and clarity** had a lift of 3.54.
- **Onboarding and guidance** had a lift of 3.16.
- **Fun and enjoyment** had a lift of only 0.50 and was strongly associated with recommended reviews.

The results reinforce the earlier text analysis. Negative feedback is more strongly associated with repetition, unclear progression, and insufficient guidance than with challenge itself.

## 6. SQL Analysis Summary

The SQL analysis reproduced and extended several findings from the Python exploratory analysis.

Key findings include:

- The English-language review dataset contains 4,511 valid reviews with a 90.73% recommendation rate.
- Review volume was highest in June 2026.
- Recommendation rate increased from 58.78% among reviews submitted before two hours of playtime to 96.19% among reviews submitted after 200 hours.
- Negative reviews were more strongly associated with repetition, onboarding problems, and unclear progression.
- The normalized database structure allowed review-level information to be joined with multiple review themes.

The SQL module demonstrates data-quality validation, conditional aggregation, segmentation, joins, CTEs, and window functions in a practical review-analysis workflow.

In [11]:
connection.close()

print("Database connection closed.")

Database connection closed.
